# SQL Queries — Group 3 Final Project

## Inflation, Interest Rates, and Market Performance
This file contains all 10 required SQL queries for the final project, separate from the main Python narrative notebook. Each query includes comments explaining **what** it does, **how** it works, and **why** it is relevant to the analysis.
**SQL tool used:** SQLite via Python's `sqlite3` module

### Query feature summary:

| Requirement | Queries |
|:---|:---|
| Joins (min 3) | Queries 1, 6, 7, 8 |
| Window functions (min 2) | Queries 7, 10 |
| GROUP BY (min 2) | Queries 1, 2, 3, 5, 6, 7, 8 |
| Subqueries (min 2) | Queries 4, 9 |

## Database Setup
The cells below recreate the in-memory SQLite database and load the same three tables used in the main notebook: `macro`, `etf_returns`, and `etf_metadata`. This allows all queries in this file to run independently.

In [13]:
import pandas as pd
import numpy as np
import requests
import time
import sqlite3
from datetime import datetime

# Download CPI data
FRED_API_KEY = "b37dc26c81cedf5337f95e307bfd8497"
fred_url = "https://api.stlouisfed.org/fred/series/observations"

cpi_raw = requests.get(fred_url, params={
    "series_id": "CPIAUCSL", "api_key": FRED_API_KEY,
    "file_type": "json", "observation_start": "2015-01-01",
    "observation_end": "2024-12-31", "frequency": "m"
}).json()

cpi = pd.DataFrame(cpi_raw["observations"])[["date", "value"]]
cpi = cpi.rename(columns={"date": "Date", "value": "CPI"})
cpi["Date"] = pd.to_datetime(cpi["Date"])
cpi["CPI"] = pd.to_numeric(cpi["CPI"], errors="coerce")
cpi["CPI_YoY"] = cpi["CPI"].pct_change(12) * 100
cpi["CPI_MoM"] = cpi["CPI"].pct_change(1) * 100
cpi["Inflation_Regime"] = pd.cut(cpi["CPI_YoY"], bins=[-np.inf, 2, 4, np.inf],
                                  labels=["Low (<2%)", "Moderate (2-4%)", "High (>4%)"])
cpi["COVID_Period"] = cpi["Date"].between("2020-01-01", "2023-12-31")

# Download Federal Funds Rate
fed_raw = requests.get(fred_url, params={
    "series_id": "FEDFUNDS", "api_key": FRED_API_KEY,
    "file_type": "json", "observation_start": "2015-01-01",
    "observation_end": "2024-12-31", "frequency": "m"
}).json()

fed = pd.DataFrame(fed_raw["observations"])[["date", "value"]]
fed = fed.rename(columns={"date": "Date", "value": "FedFundsRate"})
fed["Date"] = pd.to_datetime(fed["Date"])
fed["FedFundsRate"] = pd.to_numeric(fed["FedFundsRate"], errors="coerce")

# Download ETF prices
tickers = ["SPY","QQQ","DIA","XLE","XLF","XLK","XLU","XLP","XLY","XLB","XLI","XLV"]
period1 = int(datetime(2015,1,1).timestamp())
period2 = int(datetime(2024,12,31).timestamp())
headers = {"User-Agent": "Mozilla/5.0"}
price_data = {}
for t in tickers:
    r = requests.get(f"https://query1.finance.yahoo.com/v8/finance/chart/{t}",
                     params={"interval":"1mo","period1":period1,"period2":period2}, headers=headers)
    res = r.json()["chart"]["result"][0]
    dates = pd.to_datetime(res["timestamp"], unit="s").to_period("M").to_timestamp()
    price_data[t] = pd.Series(res["indicators"]["adjclose"][0]["adjclose"], index=dates)
    time.sleep(0.25)

etf_prices = pd.DataFrame(price_data)
etf_prices.index.name = "Date"
annual_returns = etf_prices.pct_change(12) * 100

# Build analysis_data
cpi_m = cpi.set_index("Date")[["CPI_YoY","CPI_MoM","Inflation_Regime","COVID_Period"]]
fed_m = fed.set_index("Date")[["FedFundsRate"]]
analysis_data = cpi_m.join(fed_m, how="inner").join(annual_returns, how="inner").dropna()

# Load into SQLite
conn = sqlite3.connect(":memory:")
analysis_data.reset_index().to_sql("macro", conn, index=False, if_exists="replace")

etf_tickers = ["SPY","QQQ","DIA","XLE","XLF","XLK","XLU","XLP","XLY","XLB","XLI","XLV"]
etf_returns = analysis_data[etf_tickers].reset_index().melt(
    id_vars="Date", var_name="Ticker", value_name="AnnualReturn")
etf_returns.to_sql("etf_returns", conn, index=False, if_exists="replace")

etf_metadata = pd.DataFrame({
    "Ticker": etf_tickers,
    "Sector": ["S&P 500","Nasdaq-100","Dow Jones","Energy","Financials","Technology",
               "Utilities","Consumer Staples","Consumer Discretionary","Materials","Industrials","Health Care"],
    "ETF_Type": ["Broad","Broad","Broad","Sector","Sector","Sector",
                 "Sector","Sector","Sector","Sector","Sector","Sector"]
})
etf_metadata.to_sql("etf_metadata", conn, index=False, if_exists="replace")

def run_query(sql):
    return pd.read_sql_query(sql, conn)

print("Setup complete. Tables: macro, etf_returns, etf_metadata")

Setup complete. Tables: macro, etf_returns, etf_metadata


## Query 1: Combined Dataset Validation
**What:** Counts observations per inflation regime and shows the date range for each.

**How:** Joins `etf_returns`, `macro`, and `etf_metadata` (3-table join) to confirm all tables linked correctly, then groups by inflation regime.

**Why:** Validates that the combined dataset has sufficient observations across Low, Moderate, and High inflation regimes to support meaningful comparisons. This is the first check after merging all data sources.

**Features:** JOIN (×2), GROUP BY

In [2]:
result = run_query("""
    SELECT
        m.Inflation_Regime,
        COUNT(DISTINCT e.Ticker) AS NumETFs,
        COUNT(*)                 AS NumObservations,
        MIN(e.Date)              AS EarliestDate,
        MAX(e.Date)              AS LatestDate
    FROM etf_returns e
    JOIN macro m ON e.Date = m.Date
    JOIN etf_metadata md ON e.Ticker = md.Ticker
    GROUP BY m.Inflation_Regime
    ORDER BY m.Inflation_Regime
""")
result

,Inflation_Regime,NumETFs,NumObservations,EarliestDate,LatestDate
0,High (>4%),12,312,2021-04-01 00:00:00,2023-05-01 00:00:00
1,Low (<2%),12,432,2016-01-01 00:00:00,2021-02-01 00:00:00
2,Moderate (2-4%),12,552,2016-12-01 00:00:00,2024-12-01 00:00:00


## Query 2: Inflation Regime Distribution
**What:** Counts how many months fall into each inflation regime.

**How:** Simple GROUP BY on the `Inflation_Regime` column in the `macro` table.

**Why:** Confirms there are enough months in each regime before using it as a grouping variable throughout the EDA. Mirrors the `.value_counts()` check done in Python.

**Features:** GROUP BY

**Output:** Low: 36 months, Moderate: 46 months, High: 26 months — all sufficient for comparison.

In [14]:
result = run_query("""
    SELECT
        Inflation_Regime,
        COUNT(*) AS MonthCount
    FROM macro
    GROUP BY Inflation_Regime
    ORDER BY Inflation_Regime
""")
result

,Inflation_Regime,MonthCount
0,High (>4%),26
1,Low (<2%),36
2,Moderate (2-4%),46


## Query 3: Average Inflation and Fed Funds Rate by Regime
**What:** Shows the average CPI_YoY and Fed Funds Rate for each inflation regime.

**How:** Groups the `macro` table by `Inflation_Regime` and averages both macro variables.

**Why:** Adds a numeric anchor to the correlation heatmap — confirms that higher inflation regimes correspond with higher interest rates, supporting the positive correlation observed in the EDA. Notably, the High regime has a *lower* average Fed Funds Rate than Moderate, reflecting the lag before the Fed responded.

**Features:** GROUP BY, aggregate functions (AVG, ROUND)

In [4]:
result = run_query("""
    SELECT
        Inflation_Regime,
        ROUND(AVG(CPI_YoY), 2)      AS AvgInflation,
        ROUND(AVG(FedFundsRate), 2) AS AvgFedFundsRate
    FROM macro
    GROUP BY Inflation_Regime
    ORDER BY AvgInflation
""")
result

,Inflation_Regime,AvgInflation,AvgFedFundsRate
0,Low (<2%),1.36,0.84
1,Moderate (2-4%),2.67,2.99
2,High (>4%),6.64,1.71


## Query 4: Months Where the Fed Was Behind the Curve
**What:** Identifies months where inflation was above the high-inflation average but the Fed Funds Rate was still below 2%.

**How:** Uses a WHERE-clause 

**subquery** to dynamically calculate the average CPI_YoY during high inflation periods, then filters for months where rates hadn't yet responded.

**Why:** Directly supports the finding that inflation rose before the Fed responded — isolates exactly those 9 months (Nov 2021 – Jul 2022) as concrete data points.

**Features:** Subquery (WHERE clause)

In [5]:
result = run_query("""
    SELECT
        Date,
        ROUND(CPI_YoY, 2) AS CPI_YoY,
        FedFundsRate
    FROM macro
    WHERE CPI_YoY > (SELECT AVG(CPI_YoY) FROM macro WHERE Inflation_Regime = 'High (>4%)')
      AND FedFundsRate < 2.0
    ORDER BY Date
""")
result

,Date,CPI_YoY,FedFundsRate
0,2021-11-01 00:00:00,6.90,0.08
1,2021-12-01 00:00:00,7.17,0.08
2,2022-01-01 00:00:00,7.56,0.08
3,2022-02-01 00:00:00,7.94,0.08
4,2022-03-01 00:00:00,8.57,0.20
5,2022-04-01 00:00:00,8.23,0.33
6,2022-05-01 00:00:00,8.54,0.77
7,2022-06-01 00:00:00,8.98,1.21
8,2022-07-01 00:00:00,8.46,1.68


## Query 5: Side-by-Side Annual Returns for SPY, XLE, and XLK
**What:** Shows the 12-month annual return for SPY, XLE, and XLK in a single row per month.

**How:** Uses CASE statements inside MAX() to pivot the long-format `etf_returns` table into wide format with one column per ticker, grouped by date.

**Why:** Confirms that these three ETFs move differently over time before normalizing them in Python — the divergence between XLE and XLK during the high-inflation period is visible in the raw numbers.

**Features:** GROUP BY, CASE/pivot logic

In [6]:
result = run_query("""
    SELECT
        e.Date,
        MAX(CASE WHEN e.Ticker = 'SPY' THEN ROUND(e.AnnualReturn, 2) END) AS SPY,
        MAX(CASE WHEN e.Ticker = 'XLE' THEN ROUND(e.AnnualReturn, 2) END) AS XLE,
        MAX(CASE WHEN e.Ticker = 'XLK' THEN ROUND(e.AnnualReturn, 2) END) AS XLK
    FROM etf_returns e
    GROUP BY e.Date
    ORDER BY e.Date
""")
result

,Date,SPY,XLE,XLK
0,2016-01-01 00:00:00,-0.87,-20.62,5.24
1,2016-02-01 00:00:00,-6.22,-26.24,-3.18
2,2016-03-01 00:00:00,1.61,-17.81,9.00
3,2016-04-01 00:00:00,1.09,-15.87,0.85
4,2016-05-01 00:00:00,1.51,-12.09,3.85
...,...,...,...,...
103,2024-08-01 00:00:00,26.92,6.09,26.40
104,2024-09-01 00:00:00,36.10,0.39,38.79
105,2024-10-01 00:00:00,37.81,7.57,36.51
106,2024-11-01 00:00:00,33.81,16.83,27.17


## Query 6: Average Broad Market ETF Return by Inflation Regime
**What:** Shows the average 12-month return for SPY, QQQ, and DIA broken out by inflation regime.

**How:** Joins `etf_returns`, `macro`, and `etf_metadata` (3-table join), filters to broad market ETFs, and groups by regime and ticker.

**Why:** SQL equivalent of the Python bar chart — confirms that moderate inflation is associated with the strongest broad market returns and that QQQ is the most sensitive to regime changes.

**Features:** JOIN (×2), GROUP BY, WHERE filter

**Output:** QQQ ranges from 6.89% (High) to 28.96% (Moderate); SPY and DIA are more stable across regimes.

In [15]:
result = run_query("""
    SELECT
        m.Inflation_Regime,
        e.Ticker,
        ROUND(AVG(e.AnnualReturn), 2) AS AvgAnnualReturn
    FROM etf_returns e
    JOIN macro m ON e.Date = m.Date
    JOIN etf_metadata md ON e.Ticker = md.Ticker
    WHERE md.ETF_Type = 'Broad'
    GROUP BY m.Inflation_Regime, e.Ticker
    ORDER BY m.Inflation_Regime, e.Ticker
""")
result

,Inflation_Regime,Ticker,AvgAnnualReturn
0,High (>4%),DIA,10.28
1,High (>4%),QQQ,6.89
2,High (>4%),SPY,10.61
3,Low (<2%),DIA,6.83
4,Low (<2%),QQQ,19.22
5,Low (<2%),SPY,9.01
6,Moderate (2-4%),DIA,18.99
7,Moderate (2-4%),QQQ,28.96
8,Moderate (2-4%),SPY,20.51


## Query 7: Sector ETF Return Ranking During High Inflation
**What:** Ranks all sector ETFs by average annual return during high inflation periods, from best to worst.

**How:** Filters to high inflation months, joins to `etf_metadata` for sector labels, groups by ticker, and uses the `RANK()` 

**window function** to order sectors by performance.

**Why:** Makes the sector ranking explicit and confirms that Energy (XLE) leads at 52.25% while Consumer Discretionary (XLY) lags at 4.25% during elevated inflation.

**Features:** JOIN (×2), GROUP BY, Window function (RANK)

In [16]:
result = run_query("""
    SELECT
        e.Ticker,
        md.Sector,
        ROUND(AVG(e.AnnualReturn), 2) AS AvgReturn,
        RANK() OVER (ORDER BY AVG(e.AnnualReturn) DESC) AS ReturnRank
    FROM etf_returns e
    JOIN macro m ON e.Date = m.Date
    JOIN etf_metadata md ON e.Ticker = md.Ticker
    WHERE m.Inflation_Regime = 'High (>4%)'
      AND md.ETF_Type = 'Sector'
    GROUP BY e.Ticker, md.Sector
    ORDER BY ReturnRank
""")
result

,Ticker,Sector,AvgReturn,ReturnRank
0,XLE,Energy,52.25,1
1,XLF,Financials,17.71,2
2,XLI,Industrials,13.25,3
3,XLB,Materials,13.21,4
4,XLK,Technology,11.73,5
5,XLV,Health Care,11.64,6
6,XLP,Consumer Staples,10.16,7
7,XLU,Utilities,9.93,8
8,XLY,Consumer Discretionary,4.25,9


## Query 8: Sector ETFs That Outperformed SPY by Inflation Regime
**What:** Shows which sector ETFs beat SPY's average return within each inflation regime.

**How:** Self-joins `etf_returns` to bring SPY in as a benchmark on each date, joins to `macro` and `etf_metadata`, then uses `HAVING` to filter to sectors that beat SPY on average.

**Why:** Extends the analysis from "how did sectors perform" to "which sectors actually beat the broad market" — a more actionable insight for investors. Shows that different sectors outperform in different regimes (XLE in High, XLK in Low/Moderate).

**Features:** JOIN (×3 including self-join), GROUP BY, HAVING

In [9]:
result = run_query("""
    SELECT
        m.Inflation_Regime,
        e.Ticker,
        md.Sector,
        ROUND(AVG(e.AnnualReturn), 2) AS AvgSectorReturn,
        ROUND(AVG(spy.AnnualReturn), 2) AS AvgSPY_Return
    FROM etf_returns e
    JOIN macro m ON e.Date = m.Date
    JOIN etf_metadata md ON e.Ticker = md.Ticker
    JOIN etf_returns spy ON spy.Date = e.Date AND spy.Ticker = 'SPY'
    WHERE md.ETF_Type = 'Sector'
    GROUP BY m.Inflation_Regime, e.Ticker, md.Sector
    HAVING AVG(e.AnnualReturn) > AVG(spy.AnnualReturn)
    ORDER BY m.Inflation_Regime, AvgSectorReturn DESC
""")
result

,Inflation_Regime,Ticker,Sector,AvgSectorReturn,AvgSPY_Return
0,High (>4%),XLE,Energy,52.25,10.61
1,High (>4%),XLF,Financials,17.71,10.61
2,High (>4%),XLI,Industrials,13.25,10.61
3,High (>4%),XLB,Materials,13.21,10.61
4,High (>4%),XLK,Technology,11.73,10.61
5,High (>4%),XLV,Health Care,11.64,10.61
6,Low (<2%),XLK,Technology,20.84,9.01
7,Low (<2%),XLY,Consumer Discretionary,11.44,9.01
8,Low (<2%),XLU,Utilities,10.47,9.01
9,Low (<2%),XLP,Consumer Staples,9.22,9.01


## Query 9: COVID vs. Non-COVID Macro and SPY Return Comparison
**What:** Compares average inflation, Fed Funds Rate, and SPY return between COVID-period and non-COVID-period months.

**How:** Uses a FROM-clause 

**subquery** to pre-filter `etf_returns` to SPY only and join to `macro`, then aggregates by `COVID_Period` flag in the outer query.

**Why:** Surfaces how dramatically the macro environment differed between the two periods — inflation more than doubled (2.12% → 4.52%) while SPY returns declined and DIA was hit hardest.

**Features:** Subquery (FROM clause), JOIN, GROUP BY

In [17]:
result = run_query("""
    SELECT
        covid.COVID_Period,
        ROUND(AVG(covid.CPI_YoY), 2)      AS AvgInflation,
        ROUND(AVG(covid.FedFundsRate), 2) AS AvgFedRate,
        ROUND(AVG(covid.AnnualReturn), 2) AS AvgSPY_Return
    FROM (
        SELECT m.COVID_Period, m.CPI_YoY, m.FedFundsRate, e.AnnualReturn
        FROM etf_returns e
        JOIN macro m ON e.Date = m.Date
        WHERE e.Ticker = 'SPY'
    ) AS covid
    GROUP BY covid.COVID_Period
""")
result

,COVID_Period,AvgInflation,AvgFedRate,AvgSPY_Return
0,0,2.12,2.11,15.09
1,1,4.52,1.79,13.30


## Query 10: Month-over-Month Fed Funds Rate Change with Inflation Context
**What:** Shows each month's Fed Funds Rate, the change from the prior month, CPI_YoY, and inflation regime.

**How:** Uses the `LAG()` 

**window function** to calculate the month-over-month change in the Fed Funds Rate relative to the prior month.

**Why:** Makes the Fed's tightening cycle visible row by row — the largest jumps (0.60–0.75 increases) occurred mid-2022 as the Fed scrambled to catch up with inflation. Supports the regression analysis by showing that rate *changes* may matter more than rate *levels*.

**Features:** Window function (LAG)

**Output:** 108 rows. Key observations: rates were flat near 0.08% through early 2022, then rose rapidly with the largest monthly increase of ~0.75% in mid-2022, before beginning to decline in late 2024.

In [11]:
result = run_query("""
    SELECT
        Date,
        FedFundsRate,
        ROUND(FedFundsRate - LAG(FedFundsRate) OVER (ORDER BY Date), 2) AS FedRate_Change,
        ROUND(CPI_YoY, 2) AS CPI_YoY,
        Inflation_Regime
    FROM macro
    ORDER BY Date
""")
result

,Date,FedFundsRate,FedRate_Change,CPI_YoY,Inflation_Regime
0,2016-01-01 00:00:00,0.34,NaN,1.24,Low (<2%)
1,2016-02-01 00:00:00,0.38,0.04,0.85,Low (<2%)
2,2016-03-01 00:00:00,0.36,-0.02,0.89,Low (<2%)
3,2016-04-01 00:00:00,0.37,0.01,1.17,Low (<2%)
4,2016-05-01 00:00:00,0.37,0.00,1.08,Low (<2%)
...,...,...,...,...,...
103,2024-08-01 00:00:00,5.33,0.00,2.61,Moderate (2-4%)
104,2024-09-01 00:00:00,5.13,-0.20,2.43,Moderate (2-4%)
105,2024-10-01 00:00:00,4.83,-0.30,2.58,Moderate (2-4%)
106,2024-11-01 00:00:00,4.64,-0.19,2.72,Moderate (2-4%)
